# ICBHI 2017 Challenge - Exploratory Data Analysis (EDA)
## Respiratory Disease Detection Using Classical Machine Learning

This notebook performs an in-depth exploratory data analysis on the **ICBHI 2017 Respiratory Sound Database**.

### EDA Objectives:
1. **Patient Demographics & Clinical Diagnoses**: Understand disease distribution (COPD, Healthy, URTI, Pneumonia, etc.) and patient cohorts (pediatric vs. adult).
2. **Acoustic Acquisition & Stethoscope Hardware**: Analyze chest auscultation locations, recording duration, and sampling rate heterogeneity.
3. **Adventitious Sound Annotations**: Examine respiratory cycle characteristics and class imbalance across Normal, Crackles, Wheezes, and Both.
4. **Train / Test Benchmark Split**: Validate patient-independence to avoid data leakage during model evaluation.
5. **Acoustic Representations**: Inspect time-domain waveforms and Mel-spectrograms of respiratory sounds.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 150

### 1. Load Processed Summaries and Metadata
We load the curated tables created from the 920 audio files and annotations.

In [ ]:
DATA_DIR = "../data/processed"
patients_df = pd.read_csv(os.path.join(DATA_DIR, "patients_summary.csv"))
recordings_df = pd.read_csv(os.path.join(DATA_DIR, "recordings_summary.csv"))
cycles_df = pd.read_csv(os.path.join(DATA_DIR, "cycles_summary.csv"))

print(f"Total Unique Patients: {len(patients_df)}")
print(f"Total Audio Recordings: {len(recordings_df)}")
print(f"Total Annotated Respiratory Cycles: {len(cycles_df)}")
display(patients_df.head())

### 2. Patient Demographics and Clinical Diagnoses

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Diagnosis distribution
sns.countplot(data=patients_df, y="diagnosis", order=patients_df["diagnosis"].value_counts().index, palette="viridis", ax=axes[0], hue="diagnosis", legend=False)
axes[0].set_title("A: Patient Diagnosis Distribution (N=126)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Count")

# Age by Diagnosis
sns.boxplot(data=patients_df, x="diagnosis", y="age", palette="tab10", ax=axes[1], hue="diagnosis", legend=False)
axes[1].set_title("B: Age by Diagnosis (Adult vs Pediatric Cohorts)", fontsize=12, fontweight="bold")
axes[1].tick_params(axis='x', rotation=40)
axes[1].axhline(18, color='red', linestyle='--', label='18y Adult Cutoff')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3. Auscultation Hardware & Sampling Rate Heterogeneity
Key observation for ML preprocessing: audio files are sampled at **44.1 kHz (89.6%)**, **4.0 kHz (9.8%)**, and **10.0 kHz (0.65%)** depending on the stethoscope device. Resampling to a uniform sampling rate (e.g. 4 kHz or 16 kHz) is mandatory.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=recordings_df, x="equipment", palette="crest", ax=axes[0], hue="equipment", legend=False)
axes[0].set_title("Recording Equipment / Stethoscope Type", fontsize=12, fontweight="bold")
axes[0].tick_params(axis='x', rotation=15)

sns.countplot(data=recordings_df, x="equipment", hue="sample_rate", palette="Spectral", ax=axes[1])
axes[1].set_title("Native Sampling Rate by Stethoscope", fontsize=12, fontweight="bold")
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

### 4. Adventitious Sound Annotations (4-Class Problem)
The 4 challenge classes are:
- **Normal**: Crackles=0, Wheezes=0 (52.8%)
- **Crackle**: Crackles=1, Wheezes=0 (27.0%)
- **Wheeze**: Crackles=0, Wheezes=1 (12.8%)
- **Both**: Crackles=1, Wheezes=1 (7.3%)

In [ ]:
class_order = ["Normal", "Crackle", "Wheeze", "Both"]
class_colors = ["#2b8a3e", "#e67700", "#d6336c", "#7048e8"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.countplot(data=cycles_df, x="class", order=class_order, palette=class_colors, ax=axes[0], hue="class", legend=False)
axes[0].set_title(f"4-Class Adventitious Sound Imbalance (N={len(cycles_df)} Cycles)", fontsize=12, fontweight="bold")

sns.boxplot(data=cycles_df, x="class", y="cycle_duration", order=class_order, palette=class_colors, ax=axes[1], hue="class", legend=False, showfliers=False)
axes[1].set_title("Respiratory Cycle Duration by Sound Class", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Duration (seconds)")

plt.tight_layout()
plt.show()

### 5. Challenge Train vs. Test Benchmark Split
Notice that the train/test split has 79 train patients and 49 test patients with zero patient overlap, guaranteeing patient-independent evaluation.

In [ ]:
split_class_pct = pd.crosstab(cycles_df["class"], cycles_df["split"], normalize="columns") * 100
split_class_pct = split_class_pct.reindex(class_order)

split_class_pct.plot(kind="bar", color=["#ff006e", "#3a86ff"], figsize=(8, 4.5))
plt.title("Class Distribution Consistency Between Train and Test (%)", fontsize=12, fontweight="bold")
plt.xlabel("Sound Class")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=0)
plt.legend(["Test Set", "Train Set"])
plt.tight_layout()
plt.show()